# Nhập phiếu sửa xe
Dự đoán số phiếu sau từng lần rồi nhấn Shift+Enter.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import closing
import sqlite3
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'import_csv.py').is_file())
sys.path.insert(0,str(root))
from import_csv import import_file, UPSERT
folder = TemporaryDirectory()
db = Path(folder.name)/'workshop.db'

def snapshot():
    with closing(sqlite3.connect(db)) as connection:
        return connection.execute('SELECT * FROM tickets ORDER BY ticket_code').fetchall()

def load(name):
    print(import_file(root/'examples'/name,db))
    print(snapshot())

load('first.csv')
assert len(snapshot()) == 2

UNIQUE tìm phiếu đã có. ON CONFLICT cập nhật nội dung thay đổi.

In [ ]:
print(UPSERT)
load('second.csv')
assert len(snapshot()) == 3 and snapshot()[1][-1] == 'completed'

In [ ]:
before = snapshot()
load('second.csv')
assert snapshot() == before

File lỗi ghi SC-004 trước khi gặp ngày sai. Hãy kiểm tra SC-004 biến mất sau rollback.

In [ ]:
try:
    load('invalid.csv')
except ValueError as error:
    print(error)
assert snapshot() == before
folder.cleanup()